# 31_class_conditional_routing — Reliability-matrix routing ablation

Adds only a zero-initialized `4 experts × 22 classes` reliability matrix to the exact Notebook 31 soft MoE. The ordinary sample gate remains unchanged and dataset-blind; the 88 reliability values adjust expert weights separately for each class before the result is renormalized. Stage-A and Stage-B baseline checkpoints are reused, so only Stage C is retrained.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
STUDIES = [
    ('baseline', 'config/compact_soft_moe_3seed.yaml', 'nfv3_4way_moe_soft_comparable_repro_v1'),
    ('class_conditional', 'config/compact_soft_moe_class_conditional_3seed.yaml', 'nfv3_4way_moe_soft_class_conditional_v1'),
]
EXECUTE = False
MAX_NEW_SEEDS_PER_STUDY = 3
RUN_TESTS = True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}'}
repo = Path('/content') / GITHUB_REPO; url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir(): subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else: subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None; os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR; os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'; os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Dry-run and resumable execution

At initialization, zero reliability produces the original Notebook 31 probability mixture exactly. A small `1e-4` zero-centered reliability penalty is included in the signed Stage-C contract.

In [ ]:
import torch, yaml
from training.logging_utils import run_streaming_logged
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime before executing the study.')
if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_compact_soft_moe_run.py', 'tests/test_notebook31_ablation_methods.py', 'tests/test_dataset_blind_inference.py', 'tests/test_latent_space.py', 'tests/test_out_of_core.py'], check=True)
for condition, study_config, prefix in STUDIES:
    study = yaml.safe_load(Path(study_config).read_text())
    summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary'; summary_dir.mkdir(parents=True, exist_ok=True)
    print(f'=== {condition}: architecture={study["backbone"]["architecture"]}, parameters={study["expected_total_parameters"]:,} ===')
    command = [sys.executable, '-u', '-m', 'training.compact_soft_moe_run', '--study-config', study_config, '--prefix', prefix]
    run_streaming_logged(command, str(summary_dir / 'Notebook_Dry_Run.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'31-class-route-{condition}-dry-run')
    if EXECUTE:
        run_streaming_logged([*command, '--execute', '--max-new-seeds', str(MAX_NEW_SEEDS_PER_STUDY)], str(summary_dir / 'Notebook_Training_Stream.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'31-class-route-{condition}-training')
if not EXECUTE: print('NO TRAINING WAS STARTED. Review the graph, set EXECUTE=True, and rerun this cell.')

## Paired performance comparison

In [ ]:
import pandas as pd
from IPython.display import Image, display
summaries = {condition: Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary' for condition, _, prefix in STUDIES}
complete = {name: (path / 'Final_Study_Manifest.json').is_file() for name, path in summaries.items()}; print('Completion:', complete)
if all(complete.values()):
    validation = []; test = []; classes = []; resources = []
    for name, path in summaries.items():
        for filename, target in [('Three_Seed_Guardrails.csv', validation), ('Three_Seed_Overall.csv', test), ('Three_Seed_Per_Class.csv', classes), ('Three_Seed_Resource_Accounting.csv', resources)]:
            frame = pd.read_csv(path / filename); frame.insert(0, 'condition', name); target.append(frame)
    validation = pd.concat(validation, ignore_index=True); test = pd.concat(test, ignore_index=True)
    display(validation[['condition', 'seed', 'validation_macro_f1', 'worst_dataset_macro_f1', 'rare_recall_mean']])
    pivot = validation.pivot(index='seed', columns='condition', values='validation_macro_f1'); pivot['delta_conditional_minus_baseline'] = pivot['class_conditional'] - pivot['baseline']; display(pivot)
    metrics = [c for c in ['accuracy', 'macro_f1', 'macro_precision', 'macro_recall', 'weighted_f1', 'roc_auc_ovr_macro', 'pr_auc_ovr_macro'] if c in test]
    display(test.groupby('condition')[metrics].agg(['mean', 'std']))
    print('=== Per-class F1 mean ==='); display(pd.concat(classes, ignore_index=True).pivot_table(index='class', columns='condition', values='f1', aggfunc='mean'))
    print('=== Resource accounting ==='); display(pd.concat(resources, ignore_index=True))
else: print('Comparison appears after both three-seed manifests are complete.')

## Learned expert × class reliability matrices

In [ ]:
if complete.get('class_conditional'):
    import numpy as np
    prefix = 'nfv3_4way_moe_soft_class_conditional_v1'
    expert_names = ['NF-UNSW-NB15-v3', 'NF-ToN-IoT-v3', 'NF-BoT-IoT-v3', 'NF-CICIDS2018-v3']
    for seed in (0, 1, 2):
        checkpoint = torch.load(Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / f'{prefix}_seed{seed}' / 'stage_c_full.pt', map_location='cpu')
        reliability = checkpoint['model_state']['class_reliability'].numpy()
        class_names = checkpoint['class_names']
        print(f'=== Seed {seed}: raw zero-centered reliability ===')
        display(pd.DataFrame(reliability, index=expert_names, columns=class_names))
        preference = np.exp(reliability - reliability.max(axis=0, keepdims=True)); preference /= preference.sum(axis=0, keepdims=True)
        print(f'=== Seed {seed}: reliability-only expert preference by class ===')
        display(pd.DataFrame(preference, index=expert_names, columns=class_names))
    print('The actual routing also includes each sample’s ordinary gate logits; these tables isolate the learned class effect.')

## Training and latent-space diagnostics

In [ ]:
if all(complete.values()):
    for name, path in summaries.items():
        print('===', name, 'Stage-C history ===')
        history = pd.read_csv(path / 'Training_History_3Seed.csv').query("stage == 'C'")
        columns = [c for c in ['seed', 'epoch', 'train_total_loss', 'train_reliability_penalty', 'val_macro_f1', 'best_val_macro_f1'] if c in history]
        display(history[columns]); display(pd.read_csv(path / 'Latent_Snapshot_Summary_3Seed.csv'))
        for figure in sorted((path / 'training_figures').glob('*.png')): display(Image(filename=str(figure)))